# RVC Inference - Kaggle (No UI)

**Setup awal (sekali):**
1. Verifikasi akun dengan nomor HP: https://www.kaggle.com/settings (wajib untuk internet + GPU)
2. Buat Notebook baru -> Settings (panel kanan):
   - **Accelerator: GPU T4 x2** (atau P100)
   - **Internet: On**
3. File > Import Notebook -> import `RVC_Inference_Kaggle.ipynb` ini

Kuota: ~30 jam GPU per minggu, gratis.


In [ ]:
#@title ## 1. Parameter
import os

REPO_URL = "https://github.com/aditiya-saputra/RVC-Inference.git"
REPO_DIR = "/kaggle/working/RVC"

# ---------- Voice model ----------
# Terima: URL langsung .pth/.index/.zip, ATAU URL halaman model
# HuggingFace (https://huggingface.co/user/repo) -> semua .pth+index diambil.
# Kosongkan -> otomatis dicari dari /kaggle/input (dataset Kaggle).
MODEL_URL = ""

# ---------- Input audio ----------
INPUT_MODE = "dataset"  # "dataset" atau "url"
INPUT_DATASET_PATH = "/kaggle/input"  # folder dataset berisi audio
INPUT_URL = ""  # link audio langsung (.wav/.mp3/...)

# ---------- Konversi ----------
PITCH = 0
F0_METHOD = "rmvpe"        # rmvpe / pm
SPEAKER_ID = 0
INDEX_RATE = 0.75
RMS_MIX_RATE = 0.25
PROTECT = 0.33
RESAMPLE_SR = 0
OUTPUT_FORMAT = "wav"       # wav / flac / mp3 / m4a

WORK_DIR = "/kaggle/working/rvc-work"
INPUT_DIR = WORK_DIR + "/input"
OUTPUT_DIR = "/kaggle/working/output"  # otomatis bisa di-download dari panel output
for d in (WORK_DIR, INPUT_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Parameter siap.")

In [ ]:
#@title ## 2. Cek GPU + clone repo + install dependensi
import os, subprocess, sys

!nvidia-smi -L

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# torch/torchaudio sudah tersedia di Kaggle - jangan di-upgrade.
!pip install -q faiss-cpu librosa soundfile praat-parselmouth ffmpeg-python av transformers

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Repo siap:", REPO_DIR)

In [ ]:
#@title ## 3. Download base models (HuBERT + RMVPE)
import os
os.chdir(REPO_DIR)
!python tools/download_models.py --base

In [ ]:
#@title ## 4. Voice model (.pth / .index / .zip)
import os, shutil, subprocess, glob
from pathlib import Path

os.chdir(REPO_DIR)
WEIGHTS = Path(REPO_DIR) / "assets" / "weights"
INDICES = Path(REPO_DIR) / "assets" / "indices"
WEIGHTS.mkdir(parents=True, exist_ok=True)
INDICES.mkdir(parents=True, exist_ok=True)

if MODEL_URL.strip():
    r = subprocess.run([sys.executable, "tools/download_models.py", "--url", MODEL_URL.strip()])
    if r.returncode != 0:
        raise RuntimeError("Download voice model gagal")
else:
    print("MODEL_URL kosong - mencari .pth/.index di /kaggle/input ...")
    found = False
    for src in glob.glob("/kaggle/input/**/*", recursive=True):
        p = Path(src)
        if not p.is_file():
            continue
        lower = p.name.lower()
        if lower.endswith(".zip"):
            import zipfile
            with zipfile.ZipFile(p) as zf:
                zf.extractall("/kaggle/working/_model_zip")
            for inner in Path("/kaggle/working/_model_zip").rglob("*"):
                if inner.is_file() and inner.suffix.lower() in (".pth", ".index"):
                    dst = WEIGHTS if inner.suffix.lower() == ".pth" else INDICES
                    shutil.copy2(inner, dst / inner.name)
                    print("model :", dst / inner.name)
                    found = True
        elif lower.endswith(".pth"):
            shutil.copy2(p, WEIGHTS / p.name); print("model :", WEIGHTS / p.name); found = True
        elif lower.endswith(".index"):
            shutil.copy2(p, INDICES / p.name); print("index :", INDICES / p.name); found = True
    if not found:
        raise RuntimeError(
            "Tidak menemukan .pth/.index.\n"
            "Solusi: upload model sebagai Kaggle Dataset lalu Attach ke notebook ini\n"
            "(Add Input -> dataset kamu), ATAU isi MODEL_URL di Cell 1."
        )

candidates = sorted(WEIGHTS.glob("*.pth"))
assert candidates, "Tidak ada file .pth di assets/weights!"
MODEL_PATH = str(candidates[0])
print("\nModel terpakai:", MODEL_PATH)

In [ ]:
#@title ## 5. Input audio
import os, shutil, subprocess
from pathlib import Path

AUDIO_EXT = {".wav", ".flac", ".mp3", ".m4a", ".ogg", ".opus", ".aac", ".wma", ".mp4", ".mkv", ".webm"}
src_dir = Path(INPUT_DIR)
src_dir.mkdir(parents=True, exist_ok=True)

def collect(folder):
    return [p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXT]

if INPUT_MODE == "dataset":
    for p in collect(INPUT_DATASET_PATH):
        shutil.copy2(p, src_dir / p.name)
elif INPUT_MODE == "url":
    assert INPUT_URL.strip(), "Isi INPUT_URL di Cell 1"
    name = Path(INPUT_URL.split("?")[0]).name or "input.wav"
    subprocess.run(["curl", "-L", "-o", str(src_dir / name), INPUT_URL], check=True)

audio_files = collect(src_dir)
assert audio_files, "Tidak ada audio di " + str(src_dir)
for p in audio_files:
    print("input:", p)

In [ ]:
#@title ## 6. Jalankan konversi
import os, glob, sys
os.chdir(REPO_DIR)

print("--- Speaker tersedia ---")
!python infer/cli.py --model "{MODEL_PATH}" --list-speakers

cmd = [
    sys.executable, "infer/cli.py",
    "--model", MODEL_PATH,
    "--speaker-id", str(SPEAKER_ID),
    "--input", INPUT_DIR,
    "--output", OUTPUT_DIR,
    "--pitch", str(PITCH),
    "--f0-method", F0_METHOD,
    "--index-rate", str(INDEX_RATE),
    "--rms-mix-rate", str(RMS_MIX_RATE),
    "--protect", str(PROTECT),
    "--resample-sr", str(RESAMPLE_SR),
    "--format", OUTPUT_FORMAT,
    "--overwrite",
]

if INDEX_RATE > 0:
    indices = sorted(glob.glob("assets/indices/*.index"), key=os.path.getmtime)
    if indices:
        cmd += ["--index", indices[-1]]
        print("Index terpakai:", indices[-1])
    else:
        raise RuntimeError("INDEX_RATE > 0 tapi tidak ada file .index di assets/indices")

print("\n--- Konversi ---")
import subprocess
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(
        "Inference gagal (exit %s). Penyebab umum:\n"
        "1. Baris 'rvc-cli: error: ...' di atas - baca pesannya\n"
        "2. Base model belum terunduh -> jalankan ulang Cell 3\n"
        "3. Paket belum terpasang -> jalankan ulang Cell 2" % result.returncode
    )

In [ ]:
#@title ## 7. Hasil
import glob, os
from pathlib import Path
from IPython.display import Audio, display

results = sorted(glob.glob(OUTPUT_DIR + "/**/*", recursive=True))
results = [r for r in results if Path(r).is_file() and Path(r).stat().st_size > 0]
assert results, "Tidak ada output di " + OUTPUT_DIR

for r in results:
    size_mb = Path(r).stat().st_size / 1048576
    print("output: %s (%.2f MB)" % (r, size_mb))

print()
display(Audio(results[0]))
print("\nSemua file ada di /kaggle/working/output - klik panel Output kanan untuk download,")
print("atau jalankan 'Save Version' untuk menyimpan output permanen.")